# VERA — Full Ablation Study (Language-Table, 6 conditions × 3 seeds)

### What was fixed in this version
| Root cause | Fix |
|---|---|
| Alignment loss used **frozen** CLIP 512-dim embeddings → zero gradient | Now uses **projected d_model-dim** embeddings (trainable Linear+RMSNorm) |
| `alignment_loss_coef = 0.0` in old Cell 7 → loss disabled entirely | Set to **0.15** |
| `verbalize_consequence` thresholds too high for LT rewards (0–0.2) | Lowered: moderate→0.15, small→0.02 |
| No `state_delta` → E_exp always used 2-string reward-only fallback | LT loader now computes **pseudo state_delta** from consecutive reward diffs → 6+ distinct strings |
| `state_delta` not passed from batch to model | Trainer now threads `batch["state_delta"]` → `model.forward(state_delta=...)` |

### Ablation conditions (6 core variants)
| # | Name | What's removed |
|---|---|---|
| 0 | Full VERA | — (all 5 streams active) |
| 1 | BC baseline | E_act, E_exp, history TF |
| 2 | No language feedback | E_act + E_exp (hist TF kept) |
| 3 | No E_exp only | Stream 3b consequence token |
| 4 | No E_act only | Stream 3a action narration |
| 5 | No history TF | History sub-transformer |

**Expected runtime on A100:** ~6 × 3 seeds × ~25 min = ~7–8 hours

In [ ]:
# ── Cell 1: GPU + RAM check ──────────────────────────────────────────────────
import torch, psutil, os

cuda_ok = torch.cuda.is_available()
print(f'CUDA GPU : {cuda_ok}')
if cuda_ok:
    print(f'  Device : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('  ❌  No CUDA GPU — Runtime → Change runtime type → T4 GPU')
    raise SystemExit('Switch to a GPU runtime and re-run.')

print(f'RAM  : {psutil.virtual_memory().total/1e9:.1f} GB total  |  '
      f'{psutil.virtual_memory().available/1e9:.1f} GB free')
print('GPU check passed ✓')

In [ ]:
# ── Cell 2: Mount Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/VERA_LT'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
print('Drive mounted ✓  →', DRIVE_ROOT)

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
!pip install -q git+https://github.com/openai/CLIP.git ftfy regex pyyaml
import clip, torch, yaml
print('torch:', torch.__version__, '  CLIP ready ✓')

In [ ]:
# ── Cell 4: Clone / pull repo + verify fix loaded ────────────────────────────
import os, sys

REPO_URL = 'https://github.com/sara-kaz/RLConditionedVLA.git'
REPO_DIR = '/content/RLConditionedVLA'

if os.path.exists(REPO_DIR):
    result = !cd {REPO_DIR} && git pull
    print('\n'.join(result))
else:
    !git clone -q {REPO_URL} {REPO_DIR}
    print('Cloned ✓')

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Force-clear stale module cache after git pull
for mod in list(sys.modules.keys()):
    if any(k in mod for k in ('vera_model', 'trajectory_dataset', 'sft_trainer')):
        del sys.modules[mod]

# ── Verify the E_exp fix is present ──────────────────────────────────────────
import numpy as np
from models.vera_model import verbalize_consequence

lt_rewards   = [0.061, 0.061, 0.080, 0.163, 0.080, 0.0002, 0.0002]
delta_r      = np.zeros(len(lt_rewards))
delta_r[1:]  = np.diff(lt_rewards)
state_deltas = np.clip(-delta_r * 3.0, -0.5, 0.5)

seen = set()
print('\nE_exp string diversity check (should see ≥ 5 distinct strings):')
for r, d in zip(lt_rewards, state_deltas):
    s = verbalize_consequence(float(r), float(d))
    seen.add(s)
    print(f'  r={r:.4f}  δ={d:+.3f}  →  {s}')

n_distinct = len(seen)
if n_distinct >= 5:
    print(f'\n  ✓  {n_distinct} distinct strings — fix confirmed')
else:
    print(f'\n  ❌  Only {n_distinct} strings — git pull may not have updated the file')
    print('      Re-run this cell; if still failing, check git log:')
    !cd {REPO_DIR} && git log --oneline -5

print('\nRepo:', os.getcwd())

In [ ]:
# ── Cell 5: Set data path ─────────────────────────────────────────────────────
#
# OPTION A — Real Language-Table data (recommended)
#   Set REAL_LT_ROOT to the folder that contains episode_000/, episode_001/, ...
#   Each episode folder must have a steps.pkl file.
#   If you have not downloaded real LT data, set REAL_LT_ROOT = None.
#
# OPTION B — Rendered synthetic data (fallback, generated automatically below)
#   64×64 red-block-push episodes with proper directional labels.
#   Useful for quick smoke tests on any GPU without downloading datasets.

import os, sys, pickle, tarfile, shutil
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# ── Change this line to your real LT path, or leave None ─────────────────────
REAL_LT_ROOT = None   # e.g. '/content/drive/MyDrive/language_table_data'
# REAL_LT_ROOT = '/content/drive/MyDrive/language_table_data'

# ─────────────────────────────────────────────────────────────────────────────

LOCAL_SYNTH = Path('/content/lt_rendered_synth')
SYNTH_TAR   = f'{DRIVE_ROOT}/lt_rendered_synth.tar'
TRAIN_DIR   = LOCAL_SYNTH / 'train'
VAL_DIR     = LOCAL_SYNTH / 'val'
IMG_STORE   = 64
TRAIN_N, VAL_N, EP_LEN = 900, 135, 20

DIR_NAMES = [
    'to the right', 'up and to the right', 'upward', 'up and to the left',
    'to the left', 'down and to the left', 'downward', 'down and to the right',
]

def count_ep(d):
    return len(list(Path(d).glob('episode_*'))) if Path(d).exists() else 0

if REAL_LT_ROOT and os.path.exists(REAL_LT_ROOT):
    # ── Option A: real LT data ─────────────────────────────────────────────
    LT_DATA_PATH = REAL_LT_ROOT
    USE_SYNTHETIC = False
    n_eps = len(list(Path(LT_DATA_PATH).glob('episode_*')))
    print(f'Using REAL Language-Table data: {LT_DATA_PATH}')
    print(f'  Episodes found: {n_eps}')
    if n_eps == 0:
        raise RuntimeError(f'No episode_* folders found in {LT_DATA_PATH}')
else:
    # ── Option B: rendered synthetic ──────────────────────────────────────
    USE_SYNTHETIC = True
    from PIL import Image, ImageDraw

    if count_ep(TRAIN_DIR) >= 200:
        print(f'Synthetic cache OK: train={count_ep(TRAIN_DIR)}  val={count_ep(VAL_DIR)}')
    elif os.path.exists(SYNTH_TAR):
        print('Restoring synthetic data from Drive tar...')
        with tarfile.open(SYNTH_TAR, 'r') as t:
            t.extractall('/content/')
        print(f'Restored: train={count_ep(TRAIN_DIR)}  val={count_ep(VAL_DIR)}')
    else:
        def render_frame(bx, by, gx, gy, size=IMG_STORE):
            img = Image.new('RGB', (size, size), (235, 235, 235))
            d   = ImageDraw.Draw(img)
            d.ellipse([gx-6, gy-6, gx+6, gy+6], fill=(30, 180, 30))
            d.ellipse([int(bx)-8, int(by)-8, int(bx)+8, int(by)+8], fill=(210, 40, 40))
            return np.array(img, dtype=np.uint8)

        def make_ep(ep_idx):
            T      = EP_LEN + np.random.randint(-3, 6)
            bin_id = ep_idx % 8
            instr  = f'push the block {DIR_NAMES[bin_id]}'
            mg     = 20
            bx = float(np.random.randint(mg, IMG_STORE - mg))
            by = float(np.random.randint(mg, IMG_STORE - mg))
            angle  = bin_id * (2 * np.pi / 8)
            spread = np.random.uniform(20, 28)
            gx = float(np.clip(bx + np.cos(angle) * spread, mg, IMG_STORE - mg))
            gy = float(np.clip(by + np.sin(angle) * spread, mg, IMG_STORE - mg))
            steps = []
            for _ in range(T):
                frame = render_frame(bx, by, gx, gy)
                dx = gx - bx;  dy = gy - by
                dist = float(np.sqrt(dx*dx + dy*dy)) + 1e-6
                act = np.clip(np.array(
                    [dx/dist * 0.8 + np.random.normal(0, 0.04),
                     dy/dist * 0.8 + np.random.normal(0, 0.04)], dtype=np.float32),
                    -1.0, 1.0)
                rew = float(min(1.0, max(0.0, 1.0 - dist / (IMG_STORE * 0.5))))
                steps.append({'obs': {'rgb': frame}, 'action': act,
                              'reward': rew, 'instruction': instr})
                step_px = 4.0
                bx = float(np.clip(bx + dx/dist*step_px + np.random.normal(0, 0.5),
                                   mg//2, IMG_STORE - mg//2))
                by = float(np.clip(by + dy/dist*step_px + np.random.normal(0, 0.5),
                                   mg//2, IMG_STORE - mg//2))
            return steps

        def save_split(n, start_idx, out_dir):
            Path(out_dir).mkdir(parents=True, exist_ok=True)
            for i in tqdm(range(n), desc=str(Path(out_dir).name), leave=False):
                steps = make_ep(start_idx + i)
                ep_d = Path(out_dir) / f'episode_{i:05d}'
                ep_d.mkdir(exist_ok=True)
                with open(ep_d / 'steps.pkl', 'wb') as f:
                    pickle.dump(steps, f)

        print(f'Generating {TRAIN_N}+{VAL_N} synthetic episodes...')
        save_split(TRAIN_N, 0,       TRAIN_DIR)
        save_split(VAL_N,   TRAIN_N, VAL_DIR)

        print('Archiving to Drive...')
        all_files = [p for p in LOCAL_SYNTH.rglob('*') if p.is_file()]
        local_tar = '/content/lt_synth_tmp.tar'
        with tarfile.open(local_tar, 'w') as t:
            for p in tqdm(all_files, desc='pack', leave=False):
                t.add(str(p), arcname=str(p.relative_to(LOCAL_SYNTH.parent)))
        shutil.copy2(local_tar, SYNTH_TAR)
        os.remove(local_tar)
        print(f'Saved to Drive ({os.path.getsize(SYNTH_TAR)/1e6:.0f} MB)')

    LT_DATA_PATH = str(TRAIN_DIR)
    print(f'Using SYNTHETIC data: {LT_DATA_PATH}')
    print(f'  Train episodes: {count_ep(TRAIN_DIR)}')

print('\nData path set ✓  →', LT_DATA_PATH)

In [ ]:
# ── Cell 6: Smoke test loader + verify state_delta ───────────────────────────
from data.trajectory_dataset import load_language_table, TrajectoryDataset
import psutil, torch

eps = load_language_table(LT_DATA_PATH)
assert len(eps) > 0, 'No episodes loaded — check LT_DATA_PATH'

e0 = eps[0]
print(f'Episodes loaded : {len(eps)}')
print(f'Frames shape    : {e0["frames"].shape}')
print(f'Actions shape   : {e0["actions"].shape}  (8-bin discrete)')
print(f'ActionVec shape : {e0["action_vectors"].shape}  (expect T×2)')
print(f'state_deltas    : {"YES " + str(e0["state_deltas"].shape) if "state_deltas" in e0 else "MISSING — git pull may not have worked"}')
print(f'Instruction     : {e0["instruction"]}')
print(f'Rewards range   : [{e0["rewards"].min():.4f}, {e0["rewards"].max():.4f}]')
print(f'RAM used        : {psutil.virtual_memory().used/1e9:.1f} GB')

if 'state_deltas' not in e0:
    raise RuntimeError('state_deltas missing — Cell 4 git pull did not update trajectory_dataset.py')

# Quick dataset test — one batch
ds = TrajectoryDataset(eps[:10], history_len=4, num_vis_frames=3,
                       num_actions=8, action_dim=2, img_size=64, device='cpu')
batch0 = ds[0]
print(f'\nBatch keys      : {list(batch0.keys())}')
assert 'state_delta' in batch0, 'state_delta missing from batch'
print(f'state_delta val : {batch0["state_delta"].item():.4f}')
print('Loader smoke test ✓')

In [ ]:
# ── Cell 7: Build base config ─────────────────────────────────────────────────
#
# config.yaml already has the right LT defaults baked in (action_dim=2,
# unfreeze_clip_vision=true, alignment_loss_coef=0.15, LT vocab, etc.).
# We only override the runtime paths and batch/worker settings here.

import yaml, copy

with open(f'{REPO_DIR}/configs/config.yaml') as f:
    base_cfg = yaml.safe_load(f)

# ── Data path (set by Cell 5) ─────────────────────────────────────────────────
base_cfg['data']['episodes_path'] = LT_DATA_PATH
base_cfg['data']['dataset_type']  = 'language_table'
base_cfg['data']['img_size']      = 224

# ── Runtime settings ──────────────────────────────────────────────────────────
base_cfg['training'].update({
    'output_dir'              : f'{DRIVE_ROOT}/checkpoints',
    'device'                  : 'auto',
    'batch_size'              : 32,
    'num_workers'             : 2,
    'save_every'              : 25,
})

# ── Confirm key settings ──────────────────────────────────────────────────────
print('Config ready ✓')
print(f'  episodes_path        : {base_cfg["data"]["episodes_path"]}')
print(f'  dataset_type         : {base_cfg["data"]["dataset_type"]}')
print(f'  action_dim           : {base_cfg["model"]["action_dim"]}  (expect 2 for LT)')
print(f'  unfreeze_clip_vision : {base_cfg["model"]["unfreeze_clip_vision"]}')
print(f'  alignment_loss_coef  : {base_cfg["vera"]["alignment_loss_coef"]}  (expect 0.15, NOT 0.0)')
print(f'  regression_loss_coef : {base_cfg["vera"]["regression_loss_coef"]}')
print(f'  lr                   : {base_cfg["training"]["lr"]}')
print(f'  epochs               : {base_cfg["training"]["epochs"]}')
print(f'  early_stopping       : {base_cfg["training"]["early_stopping_patience"]} epochs patience')
print(f'  clip_vision_lr       : {base_cfg["training"]["clip_vision_lr"]}')
print(f'  action_vocab entries : {len(base_cfg["vera"]["action_vocab"])}')

if base_cfg['vera']['alignment_loss_coef'] == 0.0:
    print('\n  ❌  alignment_loss_coef is 0.0 — alignment is DISABLED.')
    print('      This means git pull did not update config.yaml.')
    print('      Fix: base_cfg["vera"]["alignment_loss_coef"] = 0.15')
    base_cfg['vera']['alignment_loss_coef'] = 0.15   # force override just in case
    print('      Forced to 0.15 ✓')

In [ ]:
# ── Cell 8: Run 6 conditions × 3 seeds ───────────────────────────────────────
#
# Conditions:
#   0  full_vera    — all 5 streams (the paper's full model)
#   1  bc_baseline  — no lang feedback, no hist TF (pure BC)
#   2  no_lang      — no E_act + E_exp; hist TF + gate kept
#   3  no_exp       — no E_exp (Stream 3b); E_act kept
#   4  no_act       — no E_act (Stream 3a); E_exp kept
#   5  no_hist_tf   — no history sub-transformer; lang streams kept
#
# Each completed seed is saved to Drive and skipped on re-run.

import copy, gc, json as _json
import numpy as np, torch, yaml, psutil
from training.sft_trainer_vera import sft_train

SEEDS = [42, 123, 456]

CONDITIONS = [
    {
        'name'    : 'full_vera',
        'display' : 'Full VERA (all 5 streams)',
        'overrides': {},
    },
    {
        'name'    : 'bc_baseline',
        'display' : 'BC baseline (no lang, no hist TF)',
        'overrides': {
            ('vera', 'use_lang_feedback')    : False,
            ('vera', 'use_consequence_token'): False,
            ('vera', 'use_temporal_history') : False,
        },
    },
    {
        'name'    : 'no_lang',
        'display' : 'No language feedback (hist TF + gate kept)',
        'overrides': {
            ('vera', 'use_lang_feedback')    : False,
            ('vera', 'use_consequence_token'): False,
        },
    },
    {
        'name'    : 'no_exp',
        'display' : 'No E_exp / Stream 3b (action narration only)',
        'overrides': {
            ('vera', 'use_consequence_token'): False,
        },
    },
    {
        'name'    : 'no_act',
        'display' : 'No E_act / Stream 3a (experience token only)',
        'overrides': {
            ('vera', 'use_lang_feedback'): False,
        },
    },
    {
        'name'    : 'no_hist_tf',
        'display' : 'No history sub-transformer (flat positional hist)',
        'overrides': {
            ('vera', 'use_temporal_history'): False,
        },
    },
]

all_results = {}

for ci, cond in enumerate(CONDITIONS):
    cond_results = {}
    print(f'\n{"═"*65}')
    print(f'  [{ci+1}/{len(CONDITIONS)}]  {cond["display"]}')
    print(f'{"═"*65}')

    for seed in SEEDS:
        run_dir  = f'{DRIVE_ROOT}/checkpoints/lt_{cond["name"]}/seed{seed}'
        log_path = f'{run_dir}/sft_vera_log.json'

        # Skip if already done
        if os.path.exists(log_path):
            log = _json.load(open(log_path))
            if log:
                best = max(r['val_acc'] for r in log)
                print(f'  [SKIP] seed={seed}  best val_acc={best:.4f}')
                cond_results[f'seed{seed}'] = best
                continue

        ram_gb = psutil.virtual_memory().used / 1e9
        print(f'\n  seed={seed}  RAM={ram_gb:.1f} GB')
        os.makedirs(run_dir, exist_ok=True)

        cfg = copy.deepcopy(base_cfg)
        for (sec, key), val in cond['overrides'].items():
            cfg[sec][key] = val
        cfg['training']['output_dir'] = run_dir

        torch.manual_seed(seed)
        np.random.seed(seed)
        sft_train(cfg)

        torch.cuda.empty_cache()
        gc.collect()

        best = 0.0
        if os.path.exists(log_path):
            log = _json.load(open(log_path))
            best = max(r['val_acc'] for r in log) if log else 0.0
        cond_results[f'seed{seed}'] = best
        print(f'  ✓ seed={seed}  best val_acc={best:.4f}')

    all_results[cond['name']] = cond_results

summary = {'results': all_results}
_json.dump(summary, open(f'{DRIVE_ROOT}/checkpoints/lt_summary.json', 'w'), indent=2)
print(f'\n✓ All conditions done. Summary saved to Drive.')

In [ ]:
# ── Cell 9: Paper-ready results table ────────────────────────────────────────
import json, numpy as np

data = json.load(open(f'{DRIVE_ROOT}/checkpoints/lt_summary.json'))
res  = data['results']

ORDER = [
    ('bc_baseline', 'BC/SFT baseline              (no lang, no hist TF)'),
    ('no_lang',     'No lang feedback†             (hist TF + gate kept)'),
    ('no_hist_tf',  'No history sub-transformer    (flat positional hist)'),
    ('no_act',      'No E_act / Stream 3a          (experience token only)'),
    ('no_exp',      'No E_exp / Stream 3b          (action narration only)'),
    ('full_vera',   'Full VERA ★                   (all 5 streams)'),
]

print('─'*82)
print('Language-Table ablation — validation action-classification accuracy (%)')
print(f'{"Variant":<54} {"S42":>6} {"S123":>6} {"S456":>6}  {"Mean±Std":>12}')
print('─'*82)

vera_mu, bc_mu = None, None
for key, display in ORDER:
    if key not in res:
        print(f'  {display:<54}  (not run yet)')
        continue
    v  = [res[key].get(f'seed{s}', float('nan')) for s in [42, 123, 456]]
    mu = float(np.nanmean(v))
    sd = float(np.nanstd(v))
    if key == 'full_vera' : vera_mu = mu
    if key == 'bc_baseline': bc_mu  = mu
    vals   = '  '.join(f'{x*100:5.1f}' if not np.isnan(x) else '  ---' for x in v)
    marker = '  ← best' if key == 'full_vera' else ''
    print(f'  {display:<54}  {vals}   {mu*100:.1f}±{sd*100:.1f}{marker}')

print('─'*82)

if vera_mu is not None and bc_mu is not None:
    delta = (vera_mu - bc_mu) * 100
    sign  = '+' if delta >= 0 else ''
    print(f'  VERA vs BC delta: {sign}{delta:.2f} pp')

print()
print('† Removes E_act (Stream 3a) and E_exp (Stream 3b); hist TF and reward gate remain.')
print()
print('Copy these numbers into vera_corl.tex → Table 1 (ablations).')

In [ ]:
# ── Cell 10: Download results zip ─────────────────────────────────────────────
from google.colab import files
import shutil

shutil.make_archive('/content/vera_lt_results', 'zip', f'{DRIVE_ROOT}/checkpoints')
files.download('/content/vera_lt_results.zip')
print('Download started ✓')